In [1]:
import polars as pl

In [3]:
df = pl.read_csv("../data/Golden Standard/components_AbstRCT.csv")

In [4]:
df

,text_id,component_tokens,labels
i64,str,str,str
2614,"""AbstRCT_neoplasm_21892106""","""The overall response rate was …","""Premise"""
2615,"""AbstRCT_neoplasm_21892106""","""Grade 3/4 toxicity in both arm…","""Premise"""
2616,"""AbstRCT_neoplasm_21892106""","""There was no difference in med…","""Premise"""
2617,"""AbstRCT_neoplasm_21892106""","""Patients in arm A had longer m…","""Premise"""
2618,"""AbstRCT_neoplasm_21892106""","""but this did not reach statist…","""Premise"""
…,…,…,…
4269,"""AbstRCT_mixed_28624326""","""For the IMCT group and CPT gro…","""Premise"""
4270,"""AbstRCT_mixed_28624326""","""Both procedures produced a sta…","""Claim"""
4271,"""AbstRCT_mixed_28624326""","""eyes undergoing IMCT achieved …","""Premise"""


In [5]:
df.shape

(1660, 4)

In [7]:
df.get_column("text_id").unique()

text_id
str
"""AbstRCT_mixed_29432772"""
"""AbstRCT_neoplasm_23749688"""
"""AbstRCT_mixed_29610590"""
"""AbstRCT_neoplasm_20679600"""
"""AbstRCT_glaucoma_19383599"""
…
"""AbstRCT_glaucoma_21631670"""
"""AbstRCT_neoplasm_10506606"""
"""AbstRCT_glaucoma_20202537"""


In [10]:
df_test = pl.read_csv("components_prompt_1_test_AbstRCT.csv", separator=";")

In [11]:
df_test.get_column("text_id").unique()

text_id
str
"""AbstRCT_mixed_29141712"""
"""AbstRCT_neoplasm_20733132"""
"""AbstRCT_glaucoma_21573097"""
"""AbstRCT_mixed_29577550"""
"""AbstRCT_glaucoma_11336940"""
…
"""AbstRCT_glaucoma_21183518"""
"""AbstRCT_glaucoma_12644943"""
"""AbstRCT_neoplasm_11346336"""


In [12]:
df_test.shape

(1606, 3)

In [13]:
# Build sets of component_tokens
gold_set = set(df["component_tokens"].to_list())
test_set = set(df_test["component_tokens"].to_list())

intersection = gold_set & test_set   # true positives
missing      = gold_set - test_set   # in gold but not in test (false negatives)
extra        = test_set - gold_set   # in test but not in gold (false positives)

print(f"Gold components  : {len(gold_set)}")
print(f"Test components  : {len(test_set)}")
print(f"Intersection (TP): {len(intersection)}")
print(f"Missing (FN)     : {len(missing)}")
print(f"Extra   (FP)     : {len(extra)}")

Gold components  : 1658
Test components  : 1604
Intersection (TP): 1571
Missing (FN)     : 87
Extra   (FP)     : 33


In [14]:
# Precision, Recall, F1
tp = len(intersection)
fp = len(extra)
fn = len(missing)

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall    = tp / (tp + fn) if (tp + fn) else 0.0
f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1 Score : {f1:.3f}")

Precision: 0.979
Recall   : 0.948
F1 Score : 0.963


In [ ]:
# Inspect missing components (in gold but absent from test output)
pl.DataFrame({"component_tokens": sorted(missing)})

In [ ]:
# Inspect extra components (in test but absent from gold)
pl.DataFrame({"component_tokens": sorted(extra)})